# EDA dataset chuối

Mỗi dataset có thư mục kết quả riêng mang đúng tên nó:

```
/kaggle/working/
├── BananaID/              metadata.csv, các biểu đồ .png, BananaID_eda_report.txt
├── BananaImageBD/         ...
├── BananaRipeness_GitHub/ ...
└── fingerprints/          vân tay ảnh, dùng để so trùng chéo giữa các dataset
```

Mỗi dataset chạy đủ: phân bố lớp, kích thước, đặc trưng màu, khả năng phân biệt lớp, trùng lặp nội bộ 3 mức (MD5 / dhash khít / Hamming), ảnh mẫu, rồi so trùng chéo với các dataset khác.

## Chọn chế độ chạy

Đổi đúng **một dòng** `MODE` ở cell dưới:

| MODE | Chạy gì | Dùng khi nào |
|---|---|---|
| `'kaggle'` | BananaID + BananaImageBD | Chạy nhanh, không cần Internet |
| `'github'` | BananaRipeness_GitHub | Phải bật Settings → Internet, mất thêm vài phút tải |
| `'all'` | Cả 3 | Khi muốn một mạch từ đầu đến cuối |

Chạy `'kaggle'` trước rồi `'github'` sau trong **cùng một session** thì vẫn so trùng chéo đủ cả 3: vân tay ảnh của lần chạy trước nằm lại trong `fingerprints/`, lần sau tự đọc lên để đối chiếu. Hết session Kaggle là `/kaggle/working` bị xóa, lúc đó muốn đủ phần so chéo thì chạy `MODE = 'all'`.

## 1. Import & cấu hình

`MODE` quyết định chạy nhóm nào. Hai dataset mount sẵn trong `/kaggle/input` thuộc nhóm `kaggle`, dataset tải từ GitHub thuộc nhóm `github` (cần **Settings → Internet: On**).

Đường dẫn trong `/kaggle/input` không cần gõ tay: điền từ khóa vào `match`, cell số 4 sẽ tự dò. Nếu muốn chỉ định tay thì dán thẳng vào `dir`, `dir` được ưu tiên hơn `match`.

Mặc định lấy thư mục **ripeness chưa augment**. Bản augmented chứa nguyên si toàn bộ ảnh gốc cộng thêm các bản biến đổi, nên không dùng cả hai cùng lúc. Muốn chạy bản augmented thì đổi `match` và bật `allow_aug=True`.

`exclude` loại bỏ các thư mục con khỏi mọi bước phân tích. Repo GitHub có thư mục `varying illumination` chứa 7 bản `Real Dataset` đã chỉnh sáng bằng LIME cộng vài ảnh rời không thuộc lớp nào; để nguyên thì phần đếm trùng lặp sẽ nhảy vọt vì lý do giả tạo và sinh ra một lớp rác tên `varying illumination`. Bỏ `exclude` đi nếu muốn phân tích riêng phần ảnh đổi sáng đó.

In [ ]:
from pathlib import Path
import hashlib, json, os, re, shutil, warnings
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from PIL import Image, ImageFile
from IPython.display import display

warnings.filterwarnings('ignore')
ImageFile.LOAD_TRUNCATED_IMAGES = False
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True,
                     'grid.alpha': .3, 'font.size': 10})

MODE = 'all'

DATASETS = [
    dict(name='BananaID', group='kaggle',
         dir='',
         match=['bananaid', 'ripeness'],
         github='', paths=[], allow_aug=False, exclude=[]),
    dict(name='BananaImageBD', group='kaggle',
         dir='',
         match=['bananaimagebd', 'banana ripeness detection dataset'],
         github='', paths=[], allow_aug=False, exclude=[]),
    dict(name='BananaRipeness_GitHub', group='github',
         dir='',
         match=[],
         github='https://github.com/luischuquim/BananaRipeness',
         paths=['Banana Images/Real Dataset'],
         allow_aug=False, exclude=['varying illumination']),
]

assert MODE in ('kaggle', 'github', 'all'), "MODE phải là 'kaggle', 'github' hoặc 'all'"
ACTIVE = [d for d in DATASETS if MODE == 'all' or d['group'] == MODE]
print(f"MODE = '{MODE}' → chạy: {', '.join(d['name'] for d in ACTIVE)}")
print('Bỏ qua:', ', '.join(d['name'] for d in DATASETS if d not in ACTIVE) or '(không)')
print()

NEAR_DUP_THR = 6
SAMPLES_PER_CLASS = 8
MAX_IMAGES = 0

WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
FP_DIR = WORK_DIR / 'fingerprints'
FP_DIR.mkdir(parents=True, exist_ok=True)

EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.tif', '.tiff', '.webp'}
SPLIT_NAMES = {'train', 'training', 'test', 'testing', 'val', 'valid', 'validation'}

inp = Path('/kaggle/input')
if inp.exists():
    print('Cây thư mục /kaggle/input:')
    for p in sorted(inp.iterdir()):
        print(' ', p.name)
        for q in sorted(x for x in p.rglob('*') if x.is_dir())[:12]:
            print('   ', q.relative_to(p))
else:
    print('Không thấy /kaggle/input – đang chạy ngoài Kaggle')

## 2. Tải dataset từ GitHub

Repo `luischuquim/BananaRipeness` nặng **16,5 GB**: Real Dataset 3.495 ảnh (~337 MB), Synthetic Dataset 161.280 ảnh render (~16 GB), cộng 7 bản Real chỉnh sáng trong `varying illumination`. Chỉ Real Dataset là thứ cần cho đồ án.

`git clone` không dùng được ở đây, kể cả bản sparse checkout chỉ lấy Real Dataset: git gom toàn bộ blob vào một luồng duy nhất, đứt giữa chừng là mất sạch và không nối lại được. Thực tế gặp đúng lỗi này:

```
fetch-pack: unexpected disconnect while reading sideband packet
fatal: early EOF
```

Nên cell này **tải từng file qua HTTPS** bằng GitHub API, 8 luồng song song. Mỗi ảnh là một request riêng, nên hỏng file nào chỉ tải lại file đó chứ không mất cả mẻ. File đã tải đúng dung lượng thì bỏ qua, nghĩa là chạy lại cell sẽ **tải tiếp chỗ dở** chứ không làm lại từ đầu.

Danh sách file được cache vào `github_data/<repo>_filelist.json` để lần chạy sau không gọi lại API (GitHub giới hạn 60 request/giờ cho truy cập không đăng nhập; phần liệt kê tốn 16 request, phần tải ảnh dùng `raw.githubusercontent.com` nên không tính vào giới hạn đó).

Mặc định `paths=['Banana Images/Real Dataset']` — ảnh thật, giống Cavendish. Muốn thêm ảnh tổng hợp thì thêm `'Banana Images/Synthetic Dataset'`, nhưng 161.280 ảnh sẽ làm phần trích đặc trưng chạy rất lâu.

In [ ]:
import urllib.request, urllib.parse, time
from concurrent.futures import ThreadPoolExecutor

GH_THREADS = 8
UA = {'User-Agent': 'banana-eda-notebook'}

def imgs_in(root, exclude):
    return [p for p in root.rglob('*')
            if p.is_file() and p.suffix.lower() in EXT
            and not any(k.lower() in str(p).lower() for k in exclude)]

def gh_json(url):
    with urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=60) as r:
        return json.load(r)

def gh_walk(repo, roots):
    files, stack = [], list(roots)
    while stack:
        p = stack.pop()
        url = f'https://api.github.com/repos/{repo}/contents/' + urllib.parse.quote(p)
        try:
            items = gh_json(url)
        except Exception as e:
            raise RuntimeError(f'Không liệt kê được {p}: {e}\n'
                               'Nếu báo 403 thì đã chạm giới hạn 60 request/giờ của GitHub, '
                               'đợi một lát rồi chạy lại (danh sách đã cache sẽ không gọi lại API).')
        for x in items:
            if x['type'] == 'dir':
                stack.append(x['path'])
            elif Path(x['name']).suffix.lower() in EXT:
                files.append({'path': x['path'], 'url': x['download_url'], 'size': x.get('size', 0)})
        print(f'  đã liệt kê {len(files):,} ảnh...', end='\r')
    print()
    return files

def gh_get(f, dest):
    out = dest / f['path']
    if out.exists() and (not f['size'] or out.stat().st_size == f['size']):
        return 0
    out.parent.mkdir(parents=True, exist_ok=True)
    for k in range(4):
        try:
            with urllib.request.urlopen(urllib.request.Request(f['url'], headers=UA), timeout=120) as r:
                b = r.read()
            out.write_bytes(b)
            return len(b)
        except Exception:
            time.sleep(2 * (k + 1))
    return -1

for d in ACTIVE:
    if not d['github'] or d['dir']:
        continue
    repo = re.sub(r'https?://github\.com/|\.git$', '', d['github'].rstrip('/'))
    dest = WORK_DIR / 'github_data' / repo.split('/')[-1]
    dest.mkdir(parents=True, exist_ok=True)

    cache = dest.parent / f'{repo.split("/")[-1]}_filelist.json'
    if cache.exists():
        files = json.loads(cache.read_text())
        print(f'[{d["name"]}] dùng lại danh sách đã cache: {len(files):,} ảnh')
    else:
        print(f'[{d["name"]}] liệt kê file trong {d["paths"]}')
        files = gh_walk(repo, d['paths'])
        cache.write_text(json.dumps(files))

    files = [f for f in files if not any(k.lower() in f['path'].lower() for k in d['exclude'])]
    total_mb = sum(f['size'] for f in files) / 1e6
    print(f'Cần {len(files):,} ảnh, {total_mb:,.0f} MB')

    t0, done, failed = time.time(), 0, []
    with ThreadPoolExecutor(GH_THREADS) as ex:
        for f, n in zip(files, ex.map(lambda f: gh_get(f, dest), files)):
            done += 1
            if n < 0:
                failed.append(f)
            if done % 200 == 0 or done == len(files):
                print(f'  {done:,}/{len(files):,}  ({time.time()-t0:.0f}s, lỗi {len(failed)})')

    if failed:
        print(f'Tải lại {len(failed)} file lỗi...')
        with ThreadPoolExecutor(4) as ex:
            failed = [f for f, n in zip(failed, ex.map(lambda f: gh_get(f, dest), failed)) if n < 0]
    if failed:
        raise RuntimeError(f'{len(failed)} file vẫn tải không được sau 2 vòng – chạy lại cell này, '
                           'nó sẽ bỏ qua các file đã có và chỉ tải phần còn thiếu')

    imgs = imgs_in(dest, d['exclude'])
    print(f'{d["name"]}: {len(imgs):,} ảnh, '
          f'{sum(p.stat().st_size for p in imgs)/1e6:,.0f} MB trong {time.time()-t0:.0f}s')
    d['dir'] = os.path.commonpath([str(p.parent) for p in imgs])
    print('  →', d['dir'])

## 3. Dò đường dẫn & chốt danh sách dataset

Với mỗi dataset chưa có `dir`, duyệt các thư mục con trong `/kaggle/input` và chọn thư mục chứa nhiều ảnh nhất mà đường dẫn khớp hết các từ khóa trong `match`.

In [ ]:
def list_images(root, exclude=()):
    return sorted(p for p in root.rglob('*')
                  if p.is_file() and p.suffix.lower() in EXT
                  and not any(k.lower() in str(p).lower() for k in exclude))

def img_count(d, exclude=()):
    return len(list_images(d, exclude))

def resolve(entry):
    if entry['dir']:
        return Path(str(entry['dir']).strip().strip('"').strip("'"))
    if not inp.exists():
        return None
    cands = []
    for p in inp.rglob('*'):
        if not p.is_dir():
            continue
        low = str(p).lower()
        if not all(k.lower() in low for k in entry['match']):
            continue
        if not entry['allow_aug'] and 'augmented' in low:
            continue
        if any(k.lower() in low for k in entry['exclude']):
            continue
        direct = [x for x in p.iterdir() if x.is_file() and x.suffix.lower() in EXT]
        sub = [x for x in p.iterdir() if x.is_dir()]
        if direct and not sub:
            continue
        n = img_count(p, entry['exclude'])
        if n:
            cands.append((n, len(p.parts), p))
    if not cands:
        return None
    cands.sort(key=lambda t: (-t[0], t[1]))
    return cands[0][2]

DS = []
for d in ACTIVE:
    root = resolve(d)
    n = img_count(root, d['exclude']) if root and root.exists() else 0
    print(f'{d["name"]:<24} {n:>7,} ảnh  {root}'
          + (f'   (loại: {", ".join(d["exclude"])})' if d['exclude'] else ''))
    if root and n:
        DS.append(dict(name=d['name'], root=root, n_files=n, exclude=d['exclude']))
    else:
        print('   không tìm thấy – điền tay vào "dir" ở cell cấu hình rồi chạy lại')

assert DS, 'Chưa dataset nào được tìm thấy'
print(f'\nSẽ chạy EDA cho {len(DS)} dataset')

## 4. Các hàm dùng chung

Gom toàn bộ phần tính toán thành hàm để chạy lặp cho cả 3 dataset mà không phải sửa gì.

Đặc trưng mỗi ảnh: brightness, contrast, saturation, mean RGB, tỷ lệ pixel vàng / xanh / nâu (liên quan trực tiếp tới độ chín).

Hai hash thị giác 64-bit: `dhash` so độ sáng các pixel liền kề, `ahash` so với độ sáng trung bình. Hai hash này không đổi khi ảnh bị resize hay nén lại nên bắt được ảnh trùng mà MD5 bỏ sót.

In [ ]:
POP = np.array([bin(i).count('1') for i in range(256)], np.uint8)

def to_bytes(hex_series):
    arr = np.array([int(h, 16) for h in hex_series], dtype=np.uint64)
    return np.frombuffer(np.ascontiguousarray(arr.astype('>u8')).tobytes(),
                         np.uint8).reshape(-1, 8)

def near_pairs(hex1, hex2=None, thr=NEAR_DUP_THR, chunk=512):
    if len(hex1) == 0 or (hex2 is not None and len(hex2) == 0):
        return np.zeros((0, 3), int)
    b1 = to_bytes(hex1)
    same = hex2 is None
    b2 = b1 if same else to_bytes(hex2)
    out = []
    for s in range(0, len(b1), chunk):
        blk = b1[s:s + chunk]
        d = POP[np.bitwise_xor(blk[:, None, :], b2[None, :, :])].sum(2, dtype=np.int16)
        if same:
            for r in range(len(blk)):
                d[r, :s + r + 1] = 999
        i, j = np.where(d <= thr)
        if len(i):
            out.append(np.stack([i + s, j, d[i, j]], 1))
    return np.concatenate(out) if out else np.zeros((0, 3), int)

def clusters(n, pairs):
    parent = list(range(n))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for i, j, _ in pairs:
        a, b = find(int(i)), find(int(j))
        if a != b:
            parent[a] = b
    return pd.Series([find(i) for i in range(n)])

def parse_parts(parts):
    parts = list(parts)
    i = next((k for k, p in enumerate(parts) if p.lower() in SPLIT_NAMES), None)
    if i is None:
        if not parts:
            return 'all', 'unknown', '', ''
        return 'all', parts[-1], '/'.join(parts[:-1]), ''
    split = parts[i]
    if i + 1 < len(parts):
        return split, parts[i + 1], '/'.join(parts[:i]), '/'.join(parts[i + 2:])
    return split, (parts[i - 1] if i else 'unknown'), '/'.join(parts[:max(i - 1, 0)]), ''

def bits_to_hex(bits):
    return f'{int.from_bytes(np.packbits(bits).tobytes(), "big"):016x}'

def img_feats(path):
    with Image.open(path) as im:
        rgb = im.convert('RGB')
        a = np.asarray(rgb.resize((128, 128)), np.float32) / 255
        g9 = np.asarray(rgb.convert('L').resize((9, 8)), np.float32)
        g8 = np.asarray(rgb.convert('L').resize((8, 8)), np.float32)

    g = a.mean(2)
    hsv = np.asarray(Image.fromarray((a * 255).astype(np.uint8)).convert('HSV'), np.float32)
    hue, sat, val = hsv[:, :, 0] * 360 / 255, hsv[:, :, 1] / 255, hsv[:, :, 2] / 255
    colored = (sat > .25) & (val > .2)

    return dict(
        brightness=g.mean(), contrast=g.std(),
        R=a[:, :, 0].mean(), G=a[:, :, 1].mean(), B=a[:, :, 2].mean(),
        saturation=(a.max(2) - a.min(2)).mean(),
        hue_mean=hue[colored].mean() if colored.any() else np.nan,
        yellow_frac=(colored & (hue >= 40) & (hue < 70)).mean(),
        green_frac=(colored & (hue >= 70) & (hue < 170)).mean(),
        brown_frac=((hue >= 10) & (hue < 40) & (sat > .2) & (val <= .45)).mean(),
        dark_frac=(g < .2).mean(), bright_frac=(g > .8).mean(),
        dhash=bits_to_hex((g9[:, 1:] > g9[:, :-1]).ravel()),
        ahash=bits_to_hex((g8 > g8.mean()).ravel()))

def scan(root, out_dir, exclude=()):
    paths = list_images(root, exclude)
    if MAX_IMAGES:
        paths = paths[:MAX_IMAGES]
    rows = []
    for i, p in enumerate(paths):
        sp, cls, sub, sublbl = parse_parts(p.relative_to(root).parts[:-1])
        r = dict(path=str(p), filename=p.name, split=sp, label=cls, subset=sub,
                 sublabel=sublbl,
                 ext=p.suffix.lower(), size_kb=p.stat().st_size / 1024,
                 w=np.nan, h=np.nan, ch=np.nan, mode=None, fmt=None,
                 valid=False, error=None, md5=None)
        try:
            with Image.open(p) as im:
                im.verify()
            with Image.open(p) as im:
                r.update(w=im.width, h=im.height, mode=im.mode,
                         fmt=im.format, ch=len(im.getbands()), valid=True)
            r['md5'] = hashlib.md5(p.read_bytes()).hexdigest()
        except Exception as e:
            r['error'] = f'{type(e).__name__}: {e}'
        rows.append(r)
        if (i + 1) % 2000 == 0:
            print(f'    {i+1:,}/{len(paths):,}')
    df = pd.DataFrame(rows)
    df['ratio'] = df.w / df.h
    df['mpx'] = df.w * df.h / 1e6
    df.to_csv(out_dir / 'metadata.csv', index=False)
    return df

## 5. Chạy EDA cho từng dataset

Hàm `run_eda` làm đủ các bước cho một dataset rồi trả về kết quả để phần so trùng chéo và phần báo cáo dùng lại. Toàn bộ biểu đồ lưu vào thư mục riêng của dataset đó.

In [ ]:
def run_eda(entry):
    name, root = entry['name'], entry['root']
    out = WORK_DIR / name
    out.mkdir(parents=True, exist_ok=True)
    print(f'\n{"="*90}\n{name}  –  {root}\n{"="*90}')

    df = scan(root, out, entry.get('exclude', ()))
    ok = df[df.valid].copy().reset_index(drop=True)
    print(f'Tổng {len(df):,} file | hợp lệ {len(ok):,} | lỗi {len(df)-len(ok):,}')
    if (~df.valid).any():
        display(df.loc[~df.valid, ['filename', 'error']].head(10))

    LABELS = sorted(df.label.unique())
    cmap = plt.get_cmap('tab10')
    COLORS = {l: cmap(i % 10) for i, l in enumerate(LABELS)}

    ct = df.groupby(['split', 'label']).size().unstack(fill_value=0)
    totals = df.label.value_counts()
    imbalance = totals.max() / totals.min()
    display(ct)
    print(f'Số lớp {len(totals)} | imbalance {imbalance:.2f}x | '
          f'split: {", ".join(sorted(df.split.unique()))}')
    if df.subset.nunique() > 1:
        print('Subset:', dict(df.subset.value_counts()))
    ct.to_csv(out / 'class_counts.csv')

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    axes[0, 0].bar(totals.index, totals.values, color=[COLORS[c] for c in totals.index])
    for i, v in enumerate(totals.values):
        axes[0, 0].text(i, v, f'{v:,}', ha='center', va='bottom', fontweight='bold')
    axes[0, 0].set_title('Số lượng ảnh / lớp')
    axes[0, 0].tick_params(axis='x', rotation=20)
    axes[0, 0].yaxis.set_major_locator(MaxNLocator(integer=True))

    if df.split.nunique() > 1:
        (ct.T / ct.T.sum()).plot.bar(ax=axes[0, 1], stacked=True)
        axes[0, 1].set_title('Tỷ lệ split trong từng lớp')
        axes[0, 1].legend(fontsize=8)
    else:
        axes[0, 1].pie(totals.values, labels=totals.index, autopct='%.1f%%',
                       startangle=90, colors=[COLORS[c] for c in totals.index])
        axes[0, 1].set_title('Tỷ trọng các lớp (chưa chia split)')
        axes[0, 1].grid(False)

    ext_ct = df.ext.value_counts()
    axes[1, 0].pie(ext_ct, labels=ext_ct.index, autopct='%.1f%%', startangle=90)
    axes[1, 0].set_title('Định dạng file')
    axes[1, 0].grid(False)

    mode_ct = ok['mode'].value_counts()
    axes[1, 1].bar(mode_ct.index.astype(str), mode_ct.values, color='#8da0cb')
    for i, v in enumerate(mode_ct.values):
        axes[1, 1].text(i, v, f'{v:,}', ha='center', va='bottom', fontsize=9)
    axes[1, 1].set_title('Color mode')
    fig.suptitle(f'{name} – phân bố lớp & định dạng', fontweight='bold')
    plt.tight_layout()
    fig.savefig(out / 'class_balance.png', bbox_inches='tight')
    plt.show()

    stats_cols = ['w', 'h', 'ratio', 'mpx', 'size_kb']
    display(ok[stats_cols].describe().T.round(2))
    summary = ok.groupby('label')[stats_cols].agg(['min', 'median', 'mean', 'max'])
    summary.to_csv(out / 'size_summary.csv')
    top_res = ok.groupby(['w', 'h']).size().sort_values(ascending=False)
    print('5 kích thước phổ biến nhất:')
    print(top_res.head().to_string())

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    for ax, col, t in zip(axes.ravel(), ['w', 'h', 'ratio', 'size_kb'],
                          ['Width (px)', 'Height (px)', 'Aspect Ratio', 'File Size (KB)']):
        for lbl, grp in ok.groupby('label'):
            ax.hist(grp[col], bins=40, alpha=.55, label=lbl, color=COLORS[lbl], density=True)
        ax.set_title(t)
        ax.legend(fontsize=8)
    fig.suptitle(f'{name} – kích thước & dung lượng', fontweight='bold')
    plt.tight_layout()
    fig.savefig(out / 'size_distributions.png', bbox_inches='tight')
    plt.show()

    print('Trích đặc trưng pixel & hash thị giác...')
    feats = pd.DataFrame([img_feats(p) for p in ok.path])
    ok = pd.concat([ok, feats], axis=1)
    FEAT_COLS = [c for c in feats.columns if c not in ('dhash', 'ahash')]
    ok.to_csv(out / 'valid_with_features.csv', index=False)
    pxs = ok.groupby('label')[FEAT_COLS].agg(['mean', 'std'])
    pxs.to_csv(out / 'pixel_summary.csv')
    display(pxs.round(4))

    fig, axes = plt.subplots(2, 3, figsize=(17, 9))
    for ax, col, t in zip(axes[0], ['brightness', 'saturation', 'yellow_frac'],
                          ['Brightness', 'Saturation', 'Tỷ lệ pixel vàng']):
        for lbl, grp in ok.groupby('label'):
            ax.hist(grp[col], bins=30, alpha=.55, label=lbl, color=COLORS[lbl], density=True)
        ax.set_title(t)
        ax.legend(fontsize=8)

    bp = axes[1, 0].boxplot([ok.loc[ok.label == l, 'brown_frac'] for l in LABELS],
                            patch_artist=True)
    for patch, l in zip(bp['boxes'], LABELS):
        patch.set_facecolor(COLORS[l])
    axes[1, 0].set_xticks(range(1, len(LABELS) + 1))
    axes[1, 0].set_xticklabels(LABELS, rotation=20)
    axes[1, 0].set_title('Tỷ lệ pixel nâu (vết chín) theo lớp')

    ok.groupby('label')[['R', 'G', 'B']].mean().plot.bar(
        ax=axes[1, 1], color=['#e41a1c', '#4daf4a', '#377eb8'])
    axes[1, 1].set_title('Mean RGB / lớp')
    axes[1, 1].set_ylim(0, 1)
    axes[1, 1].tick_params(axis='x', rotation=20)

    corr = ok[FEAT_COLS].corr()
    im_ax = axes[1, 2].imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
    axes[1, 2].set_xticks(range(len(FEAT_COLS)))
    axes[1, 2].set_yticks(range(len(FEAT_COLS)))
    axes[1, 2].set_xticklabels(FEAT_COLS, rotation=45, ha='right', fontsize=8)
    axes[1, 2].set_yticklabels(FEAT_COLS, fontsize=8)
    axes[1, 2].set_title('Feature Correlation')
    axes[1, 2].grid(False)
    fig.colorbar(im_ax, ax=axes[1, 2], fraction=.046)
    fig.suptitle(f'{name} – đặc trưng pixel', fontweight='bold')
    plt.tight_layout()
    fig.savefig(out / 'pixel_analysis.png', bbox_inches='tight')
    plt.show()

    sep = []
    for c in FEAT_COLS:
        gm = ok.groupby('label')[c].mean()
        within = ok.groupby('label')[c].std().mean()
        sep.append(dict(feature=c, between_std=gm.std(), within_std=within,
                        score=gm.std() / within if within else np.nan,
                        lo_class=gm.idxmin(), lo=gm.min(),
                        hi_class=gm.idxmax(), hi=gm.max()))
    sep = pd.DataFrame(sep).sort_values('score', ascending=False).reset_index(drop=True)
    sep.to_csv(out / 'feature_separability.csv', index=False)
    display(sep.round(4))

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.barh(sep.feature[::-1], sep.score[::-1], color='#66c2a5')
    ax.set_title(f'{name} – khả năng phân biệt lớp của từng đặc trưng (cao = tốt)')
    plt.tight_layout()
    fig.savefig(out / 'feature_separability.png', bbox_inches='tight')
    plt.show()

    dups = ok[ok.duplicated('md5', keep=False)].sort_values('md5')
    n_md5_dup_groups = int((ok.groupby('md5').size() > 1).sum())
    n_unique_md5 = int(ok.md5.nunique())
    leak_groups = int((ok.groupby('md5').split.nunique() > 1).sum())
    conflict_groups = int((ok.groupby('md5').label.nunique() > 1).sum())
    dhash_dup_groups = int((ok.groupby('dhash').size() > 1).sum())
    n_unique_dhash = int(ok.dhash.nunique())

    print(f'MD5 trùng: {n_md5_dup_groups} nhóm / {len(dups):,} file → còn {n_unique_md5:,} unique '
          f'({n_unique_md5/len(ok)*100:.1f}%)')
    print(f'dhash trùng khít: {dhash_dup_groups} nhóm → {n_unique_dhash:,} ảnh khác nhau về thị giác')
    print(f'Nhóm MD5 nằm ở nhiều split (leakage): {leak_groups}')
    print(f'Nhóm MD5 bị gán nhiều nhãn: {conflict_groups}')

    pairs = near_pairs(ok.dhash.values)
    n_visual_unique = int(clusters(len(ok), pairs).nunique())
    print(f'Cặp gần trùng (Hamming ≤ {NEAR_DUP_THR}): {len(pairs):,} → '
          f'{n_visual_unique:,} ảnh thực sự khác nhau ({n_visual_unique/len(ok)*100:.1f}%)')

    if len(dups):
        dups.to_csv(out / 'duplicates.csv', index=False)
    if len(pairs):
        nd = pd.DataFrame({
            'dist': pairs[:, 2],
            'file_a': ok.filename.values[pairs[:, 0]], 'label_a': ok.label.values[pairs[:, 0]],
            'split_a': ok.split.values[pairs[:, 0]], 'path_a': ok.path.values[pairs[:, 0]],
            'file_b': ok.filename.values[pairs[:, 1]], 'label_b': ok.label.values[pairs[:, 1]],
            'split_b': ok.split.values[pairs[:, 1]], 'path_b': ok.path.values[pairs[:, 1]]})
        nd['same_label'] = nd.label_a == nd.label_b
        nd = nd.sort_values('dist').reset_index(drop=True)
        nd.to_csv(out / 'near_duplicates.csv', index=False)
        print(f'Cặp gần trùng nhưng KHÁC NHÃN: {int((~nd.same_label).sum()):,}')
        display(nd.head(10)[['dist', 'label_a', 'file_a', 'label_b', 'file_b']])
    else:
        nd = pd.DataFrame(columns=['dist', 'label_a', 'label_b', 'path_a', 'path_b',
                                   'file_a', 'file_b', 'same_label'])

    fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
    per_label = pd.DataFrame({
        'tổng': ok.groupby('label').size(),
        'unique MD5': ok.groupby('label').md5.nunique(),
        'unique dhash': ok.groupby('label').dhash.nunique()})
    per_label.plot.bar(ax=axes[0], color=['#bbbbbb', '#66c2a5', '#fc8d62'])
    axes[0].set_title('Tổng vs unique theo lớp')
    axes[0].tick_params(axis='x', rotation=20)
    axes[0].legend(fontsize=8)

    if len(nd):
        axes[1].hist(nd.dist, bins=range(0, NEAR_DUP_THR + 2),
                     color='#e78ac3', align='left', rwidth=.8)
        axes[1].set_title(f'Phân bố Hamming (≤ {NEAR_DUP_THR})')
        axes[1].set_xlabel('Hamming distance')
    else:
        axes[1].text(.5, .5, 'Không có cặp gần trùng', ha='center', va='center')
        axes[1].set_title('Gần trùng')

    sizes = [n_visual_unique, len(ok) - n_visual_unique]
    axes[2].pie(sizes, labels=[f'Khác nhau\n{sizes[0]:,}', f'Trùng/gần trùng\n{sizes[1]:,}'],
                autopct='%.1f%%', startangle=90, colors=['#66c2a5', '#fc8d62'])
    axes[2].set_title('Tỷ lệ ảnh thực sự khác nhau')
    axes[2].grid(False)
    fig.suptitle(f'{name} – trùng lặp nội bộ', fontweight='bold')
    plt.tight_layout()
    fig.savefig(out / 'duplicate_analysis.png', bbox_inches='tight')
    plt.show()

    if len(nd):
        show = nd.head(3)
        fig, axes = plt.subplots(len(show), 2, figsize=(7, 3.4 * len(show)))
        axes = np.atleast_2d(axes)
        for r, (_, row) in enumerate(show.iterrows()):
            for c, (pth, lb, fn) in enumerate([(row.path_a, row.label_a, row.file_a),
                                               (row.path_b, row.label_b, row.file_b)]):
                with Image.open(pth) as im:
                    axes[r, c].imshow(im.convert('RGB'))
                axes[r, c].set_title(f'{lb} / {fn}\nHamming = {row.dist}', fontsize=8)
                axes[r, c].axis('off')
        fig.suptitle(f'{name} – các cặp gần trùng nhất', fontweight='bold')
        plt.tight_layout()
        fig.savefig(out / 'duplicate_samples.png', bbox_inches='tight')
        plt.show()

    rng = np.random.default_rng(231)
    for lbl in LABELS:
        sub = ok[ok.label == lbl]
        if not len(sub):
            continue
        idx = rng.choice(len(sub), min(SAMPLES_PER_CLASS, len(sub)), replace=False)
        samples = sub.iloc[idx]
        fig, axes = plt.subplots(2, 4, figsize=(14, 7))
        for ax, (_, row) in zip(axes.ravel(), samples.iterrows()):
            with Image.open(row.path) as im:
                ax.imshow(im.convert('RGB'))
            ax.set_title(f'{row.filename}\n{int(row.w)}×{int(row.h)}', fontsize=8)
            ax.axis('off')
        for ax in axes.ravel()[len(samples):]:
            ax.axis('off')
        fig.suptitle(f'{name} – samples: {lbl}', fontweight='bold')
        plt.tight_layout()
        fig.savefig(out / f'samples_{re.sub(r"[^0-9A-Za-z_-]+", "_", lbl)}.png',
                    bbox_inches='tight')
        plt.show()

    fp = ok[['filename', 'split', 'label', 'md5', 'dhash', 'ahash', 'w', 'h', 'path']].copy()
    fp.insert(0, 'dataset', name)
    fp.to_csv(FP_DIR / f'{name}_fingerprints.csv', index=False)

    return dict(name=name, root=root, out=out, exclude=list(entry.get('exclude', ())),
                df=df, ok=ok, labels=LABELS,
                ct=ct, totals=totals, imbalance=imbalance, sep=sep, nd=nd,
                dups=dups, top_res=top_res, feat_cols=FEAT_COLS,
                n_md5_dup_groups=n_md5_dup_groups, n_unique_md5=n_unique_md5,
                n_unique_dhash=n_unique_dhash, n_visual_unique=n_visual_unique,
                leak_groups=leak_groups, conflict_groups=conflict_groups)

In [ ]:
RES = {}
for entry in DS:
    RES[entry['name']] = run_eda(entry)
print('\nXong EDA cho:', ', '.join(RES))

## 6. So trùng chéo giữa các dataset

So từng cặp dataset: các dataset vừa chạy ở trên, cộng cả những dataset đã chạy trước đó trong cùng session (đọc lại từ `fingerprints/`). Vẫn ba mức như phần nội bộ: MD5 trùng byte, dhash trùng khít (ảnh đã resize/nén lại), Hamming ≤ ngưỡng (gần giống).

Bảng chéo nhãn cho biết ảnh trùng giữa hai dataset có được gán cùng mức độ chín hay không — nếu lệch nhau thì tiêu chuẩn gán nhãn của hai nguồn khác nhau, đó là thông tin quan trọng hơn cả con số trùng.

In [ ]:
CROSS, CROSS_DETAIL = [], []

SRC = {k: v['ok'] for k, v in RES.items()}
for f in sorted(FP_DIR.glob('*_fingerprints.csv')):
    nm = f.stem.replace('_fingerprints', '')
    if nm not in SRC:
        SRC[nm] = pd.read_csv(f)
        print(f'Nạp vân tay lần chạy trước: {nm} ({len(SRC[nm]):,} ảnh)')

names = list(SRC)
if len(names) < 2:
    print('Chỉ có 1 dataset trong session này – chưa có gì để so chéo.')
    print("Chạy tiếp MODE còn lại trong CÙNG session, hoặc dùng MODE = 'all'.")

for i in range(len(names)):
    for j in range(i + 1, len(names)):
        if names[i] not in RES and names[j] not in RES:
            continue
        a = RES.get(names[i], {'name': names[i]})
        b = RES.get(names[j], {'name': names[j]})
        a.setdefault('name', names[i])
        b.setdefault('name', names[j])
        oa, ob = SRC[names[i]], SRC[names[j]]
        md5_hit = int(len(set(oa.md5) & set(ob.md5)))
        dhash_hit = int(len(set(oa.dhash) & set(ob.dhash)))
        p = near_pairs(oa.dhash.values, ob.dhash.values)
        ma = len(set(p[:, 0])) if len(p) else 0
        mb = len(set(p[:, 1])) if len(p) else 0

        print(f'\n--- {a["name"]} ({len(oa):,})  vs  {b["name"]} ({len(ob):,}) ---')
        print(f'Trùng MD5: {md5_hit} hash | trùng dhash khít: {dhash_hit} hash | '
              f'cặp gần trùng: {len(p):,}')
        print(f'→ {ma:,}/{len(oa):,} ({ma/len(oa)*100:.2f}%) ảnh bên {a["name"]} có ảnh giống bên kia')
        print(f'→ {mb:,}/{len(ob):,} ({mb/len(ob)*100:.2f}%) ảnh bên {b["name"]} có ảnh giống bên kia')

        CROSS.append(dict(dataset_a=a['name'], dataset_b=b['name'],
                          n_a=len(oa), n_b=len(ob),
                          trung_md5=md5_hit, trung_dhash=dhash_hit,
                          cap_gan_trung=len(p),
                          anh_trung_a=ma, anh_trung_b=mb,
                          pct_a=round(ma / len(oa) * 100, 2),
                          pct_b=round(mb / len(ob) * 100, 2)))

        if len(p):
            det = pd.DataFrame({
                'dataset_a': a['name'], 'dataset_b': b['name'], 'dist': p[:, 2],
                'file_a': oa.filename.values[p[:, 0]], 'label_a': oa.label.values[p[:, 0]],
                'path_a': oa.path.values[p[:, 0]],
                'file_b': ob.filename.values[p[:, 1]], 'label_b': ob.label.values[p[:, 1]],
                'path_b': ob.path.values[p[:, 1]]})
            CROSS_DETAIL.append(det)
            print('Bảng chéo nhãn của các cặp trùng:')
            display(pd.crosstab(det.label_a, det.label_b))

CROSS = pd.DataFrame(CROSS)
display(CROSS)

CD = pd.concat(CROSS_DETAIL).sort_values('dist').reset_index(drop=True) if CROSS_DETAIL \
     else pd.DataFrame(columns=['dataset_a', 'dataset_b', 'dist', 'label_a', 'label_b',
                                'file_a', 'file_b', 'path_a', 'path_b'])

for r in RES.values():
    if len(CROSS):
        CROSS.to_csv(r['out'] / 'cross_dataset_overlap.csv', index=False)
    if len(CD):
        CD.to_csv(r['out'] / 'cross_dataset_pairs.csv', index=False)

if len(CD):
    show = CD[CD.path_a.apply(lambda x: Path(str(x)).exists())
              & CD.path_b.apply(lambda x: Path(str(x)).exists())].head(4)
if len(CD) and len(show):
    fig, axes = plt.subplots(len(show), 2, figsize=(7, 3.4 * len(show)))
    axes = np.atleast_2d(axes)
    for r, (_, row) in enumerate(show.iterrows()):
        for c, (pth, tag) in enumerate([(row.path_a, f'{row.dataset_a} / {row.label_a}'),
                                        (row.path_b, f'{row.dataset_b} / {row.label_b}')]):
            with Image.open(pth) as im:
                axes[r, c].imshow(im.convert('RGB'))
            axes[r, c].set_title(f'{tag}\nHamming = {row.dist}', fontsize=8)
            axes[r, c].axis('off')
    fig.suptitle('Các cặp ảnh giống nhau giữa 2 dataset khác nguồn', fontweight='bold')
    plt.tight_layout()
    for r in RES.values():
        fig.savefig(r['out'] / 'cross_dataset_samples.png', bbox_inches='tight')
    plt.show()
else:
    print('\nKhông có cặp ảnh nào gần trùng giữa các dataset.')

## 7. Biểu đồ so sánh các dataset

Đặt các dataset vừa chạy cạnh nhau: quy mô, số lớp, mức mất cân bằng, tỷ lệ dư thừa và mức trùng chéo.

In [ ]:
def cmp_row(r):
    nd, n = r['nd'], len(r['ok'])
    xlab = round((~nd.same_label).sum() / len(nd) * 100, 1) if len(nd) else 0.0
    best = r['sep'].iloc[0]
    return {
        'dataset': r['name'], 'tổng ảnh': len(r['df']), 'hợp lệ': n,
        'số lớp': len(r['labels']), 'imbalance': round(r['imbalance'], 2),
        'unique sau dedup': r['n_visual_unique'],
        'dư thừa %': round((1 - r['n_visual_unique'] / n) * 100, 1),
        'cặp gần trùng khác nhãn %': xlab,
        'đặc trưng tách lớp tốt nhất': best.feature,
        'điểm tách lớp': round(best.score, 2),
        'có split sẵn': 'có' if r['df'].split.nunique() > 1 else 'không',
        'leakage split': r['leak_groups'],
        'kích thước phổ biến': f"{int(r['top_res'].index[0][0])}x{int(r['top_res'].index[0][1])}",
    }

cmp = pd.DataFrame([cmp_row(r) for r in RES.values()])
display(cmp)
cmp.to_csv(WORK_DIR / 'so_sanh_dataset.csv', index=False)
for r in RES.values():
    cmp.to_csv(r['out'] / 'dataset_comparison.csv', index=False)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
x = np.arange(len(cmp))
axes[0].bar(x - .2, cmp['hợp lệ'], .4, label='ảnh hợp lệ', color='#bbbbbb')
axes[0].bar(x + .2, cmp['unique sau dedup'], .4, label='sau khi dedup', color='#66c2a5')
axes[0].set_xticks(x)
axes[0].set_xticklabels(cmp.dataset, rotation=15, fontsize=8)
axes[0].set_title('Quy mô trước / sau khi loại trùng')
axes[0].legend(fontsize=8)

axes[1].bar(x, cmp['dư thừa %'], color='#fc8d62')
for i, v in enumerate(cmp['dư thừa %']):
    axes[1].text(i, v, f'{v}%', ha='center', va='bottom', fontsize=9)
axes[1].set_xticks(x)
axes[1].set_xticklabels(cmp.dataset, rotation=15, fontsize=8)
axes[1].set_title('Tỷ lệ ảnh dư thừa trong nội bộ')

if len(CROSS):
    lbl = [f'{r.dataset_a}\nvs\n{r.dataset_b}' for _, r in CROSS.iterrows()]
    axes[2].bar(range(len(CROSS)), CROSS[['pct_a', 'pct_b']].max(axis=1), color='#e78ac3')
    for i, v in enumerate(CROSS[['pct_a', 'pct_b']].max(axis=1)):
        axes[2].text(i, v, f'{v}%', ha='center', va='bottom', fontsize=9)
    axes[2].set_xticks(range(len(CROSS)))
    axes[2].set_xticklabels(lbl, fontsize=7)
    axes[2].set_title('Mức trùng chéo cao nhất giữa từng cặp')
else:
    axes[2].text(.5, .5, 'Chỉ có 1 dataset', ha='center', va='center')

plt.tight_layout()
for r in RES.values():
    fig.savefig(r['out'] / 'dataset_comparison.png', bbox_inches='tight')
plt.show()

## 8. Xuất báo cáo cho từng dataset

Mỗi dataset nhận một file `<tên_dataset>_eda_report.txt` trong thư mục riêng, gồm cả phần so trùng với 2 dataset còn lại.

In [ ]:
def build_report(r, cross):
    name, df, ok = r['name'], r['df'], r['ok']
    LABELS, totals, ct, sep, nd = r['labels'], r['totals'], r['ct'], r['sep'], r['nd']
    n = len(ok)

    def pc(x, total):
        return f'{x/total*100:.1f}%' if total else '-'

    L, A = [], None
    A = L.append
    A('Báo cáo EDA dataset – phân loại độ chín của chuối')
    A(f'Dataset : {name}')
    A(f'Đường dẫn: {r["root"]}')
    if r['exclude']:
        A(f'Đã loại khỏi phân tích: {", ".join(r["exclude"])}')
    A(f'Thời điểm: {datetime.now():%Y-%m-%d %H:%M}')
    A('')

    A('TỔNG QUAN DATASET')
    A(f'- Tổng số file ảnh: {len(df):,}')
    A(f'- Ảnh hợp lệ: {n:,} ({pc(n, len(df))})' +
      ('' if n == len(df) else f' – {len(df)-n:,} file lỗi/không đọc được'))
    A(f'- Số lớp: {len(LABELS)} ({", ".join(map(str, LABELS))})')
    if df.subset.nunique() > 1:
        A(f'- Dataset có {df.subset.nunique()} nhóm con: ' +
          ', '.join(f'{k} {v:,}' for k, v in df.subset.value_counts().items()))
    if df.sublabel.replace('', np.nan).notna().any():
        sl = df.loc[df.sublabel != '', 'sublabel'].value_counts()
        A(f'- Trong mỗi lớp còn chia tiếp {len(sl)} mức: ' +
          ', '.join(f'{k} {v:,}' for k, v in sl.items()))
    for e, c in df.ext.value_counts().items():
        A(f'- Định dạng {e}: {c:,} ({pc(c, len(df))})')
    for m, c in ok['mode'].value_counts().items():
        A(f'- Color mode {m}: {c:,} ({pc(c, n)})')
    A('')

    A('PHÂN BỐ LỚP & CÂN BẰNG')
    for l, c in totals.items():
        A(f'- {l}: {c:,} ảnh ({pc(c, len(df))})')
    A(f'- Tỷ lệ mất cân bằng: {r["imbalance"]:.2f}x '
      f'({totals.idxmax()} nhiều nhất, {totals.idxmin()} ít nhất)')
    A('- Mức mất cân bằng ' + ('KHÔNG đáng kể' if r['imbalance'] < 1.2 else
                               'NHẸ, chấp nhận được nhưng nên dùng class_weight' if r['imbalance'] < 1.5 else
                               'ĐÁNG KỂ, nên dùng class_weight hoặc oversampling' if r['imbalance'] < 3 else
                               'NẶNG, cần cân bằng lại dữ liệu trước khi train'))
    A('')

    A('PHÂN BỐ SPLIT')
    if df.split.nunique() > 1:
        for sp, c in df.split.value_counts().items():
            detail = ' / '.join(f'{l} {v:,}' for l, v in ct.loc[sp][ct.loc[sp] > 0].items())
            A(f'- {sp}: {c:,} ảnh ({pc(c, len(df))}) – {detail}')
        A(f'- Số nhóm MD5 xuất hiện ở nhiều split: {r["leak_groups"]}' +
          (' – có DATA LEAKAGE giữa các split' if r['leak_groups'] else ' – không có leakage theo MD5'))
    else:
        A('- Dataset chưa chia sẵn train/test, ảnh nằm chung theo lớp')
        A('- Cần tự chia train/val/test (stratified theo nhãn) sau khi đã loại ảnh trùng')
    A('')

    A('TRÙNG LẶP TRONG DATASET')
    A(f'- Trùng từng byte (MD5): {r["n_md5_dup_groups"]} nhóm / {len(r["dups"]):,} file')
    A(f'- Số ảnh unique theo MD5: {r["n_unique_md5"]:,}/{n:,} ({pc(r["n_unique_md5"], n)})')
    A(f'- Số ảnh khác nhau theo dhash: {r["n_unique_dhash"]:,} (bắt cả ảnh đã resize/nén lại)')
    A(f'- Cặp gần trùng Hamming ≤ {NEAR_DUP_THR}: {len(nd):,}')
    A(f'- Sau khi gộp cụm gần trùng còn {r["n_visual_unique"]:,} ảnh thực sự khác nhau '
      f'({pc(r["n_visual_unique"], n)})')
    dup_rate = 1 - r['n_visual_unique'] / n
    A(f'- Tỷ lệ dư thừa tổng thể: {dup_rate*100:.1f}%' +
      (' – không đáng kể' if dup_rate < .02 else
       ' – ở mức chấp nhận được, nên lọc trước khi train' if dup_rate < .10 else
       ' – CAO, bắt buộc deduplicate trước khi chia tập'))
    if len(nd):
        n_conf = int((~nd.same_label).sum())
        A(f'- Cặp gần trùng nhưng gán KHÁC NHÃN: {n_conf:,}' +
          (' – cần soát lại nhãn ở các cặp này' if n_conf else ''))
    A(f'- Nhóm MD5 bị gán nhiều nhãn khác nhau: {r["conflict_groups"]}')
    A('')

    A('TRÙNG LẶP VỚI CÁC DATASET KHÁC')
    mine = cross[(cross.dataset_a == name) | (cross.dataset_b == name)] if len(cross) else cross
    if len(mine):
        worst = 0
        for _, c in mine.iterrows():
            other = c.dataset_b if c.dataset_a == name else c.dataset_a
            n_other = c.n_b if c.dataset_a == name else c.n_a
            hit = c.anh_trung_a if c.dataset_a == name else c.anh_trung_b
            pct = c.pct_a if c.dataset_a == name else c.pct_b
            worst = max(worst, pct)
            A(f'- vs {other} ({n_other:,} ảnh): trùng MD5 {c.trung_md5} hash, '
              f'trùng dhash {c.trung_dhash} hash, {c.cap_gan_trung:,} cặp gần trùng')
            A(f'  → {hit:,}/{n:,} ảnh ({pct}%) của {name} có ảnh giống bên {other}')
        A(f'- Mức trùng cao nhất với một dataset khác: {worst}%')
        A('- Đánh giá: ' + ('các dataset gần như độc lập, gộp lại được mà không sợ lặp dữ liệu' if worst < 1 else
                            'trùng ít, gộp được nhưng nên dedup theo dhash sau khi gộp' if worst < 10 else
                            'TRÙNG NHIỀU, nếu gộp sẽ có nhiều ảnh lặp – nên chọn 1 dataset làm chính hoặc dedup kỹ' if worst < 30 else
                            'TRÙNG RẤT NHIỀU, hai dataset gần như cùng một nguồn ảnh'))
    else:
        A('- Không có dataset nào khác trong lần chạy này để đối chiếu')
    A('')

    A('KÍCH THƯỚC ẢNH')
    A(f'- Width:  min {ok.w.min():.0f} – max {ok.w.max():.0f} – mean {ok.w.mean():.0f} – median {ok.w.median():.0f}')
    A(f'- Height: min {ok.h.min():.0f} – max {ok.h.max():.0f} – mean {ok.h.mean():.0f} – median {ok.h.median():.0f}')
    top = r['top_res']
    A(f'- Số kích thước khác nhau: {len(top):,}')
    A(f'- Kích thước phổ biến nhất: {int(top.index[0][0])}×{int(top.index[0][1])} '
      f'({top.iloc[0]:,} ảnh, {pc(top.iloc[0], n)})')
    A('- ' + ('Kích thước gần như ĐỒNG NHẤT, resize sẽ ít ảnh hưởng' if top.iloc[0] / n > .9 else
             'Kích thước KHÔNG đồng nhất → cần resize thống nhất (224×224 hoặc 256×256)'))
    A(f'- Aspect ratio: mean {ok.ratio.mean():.2f}, phạm vi {ok.ratio.min():.2f} – {ok.ratio.max():.2f}')
    A('- ' + ('Phần lớn ảnh gần vuông → resize vuông ít biến dạng' if abs(ok.ratio.mean() - 1) < .15 else
             'Ảnh lệch khỏi hình vuông → cân nhắc pad thay vì resize thẳng'))
    A(f'- Dung lượng: mean {ok.size_kb.mean():.0f} KB, median {ok.size_kb.median():.0f} KB, '
      f'phạm vi {ok.size_kb.min():.0f} – {ok.size_kb.max():.0f} KB')
    A('')

    A('ĐẶC TRƯNG PIXEL & MÀU SẮC THEO LỚP')
    for l in LABELS:
        g = ok[ok.label == l]
        if not len(g):
            continue
        A(f'- {l} ({len(g):,} ảnh):')
        A(f'    brightness {g.brightness.mean():.3f} ± {g.brightness.std():.3f} | '
          f'contrast {g.contrast.mean():.3f} | saturation {g.saturation.mean():.3f}')
        A(f'    RGB = {g.R.mean():.3f} / {g.G.mean():.3f} / {g.B.mean():.3f}')
        A(f'    pixel vàng {g.yellow_frac.mean()*100:.1f}% | xanh {g.green_frac.mean()*100:.1f}% | '
          f'nâu {g.brown_frac.mean()*100:.1f}%')
    A('')

    A('ĐẶC TRƯNG PHÂN BIỆT LỚP TỐT NHẤT')
    for _, s in sep.head(5).iterrows():
        A(f'- {s.feature}: score {s.score:.2f} (thấp nhất ở {s.lo_class} = {s.lo:.3f}, '
          f'cao nhất ở {s.hi_class} = {s.hi:.3f})')
    A(f'- Rõ nhất là {sep.iloc[0].feature}; nếu các đặc trưng màu tách được lớp thì feature thủ công '
      'còn dùng được, nếu score đều thấp thì nên đi thẳng CNN/transfer learning')
    A('')

    A('NHẬN XÉT CHUNG VÀ GỢI Ý')
    A(f'- Quy mô sau khi loại trùng: {r["n_visual_unique"]:,} ảnh cho {len(LABELS)} lớp '
      f'(~{r["n_visual_unique"]//max(len(LABELS),1):,} ảnh/lớp)')
    A('- ' + ('Số lượng khá ít, nên dùng transfer learning + augmentation mạnh'
             if r['n_visual_unique'] < 2000 else
             'Quy mô vừa, đủ cho fine-tuning CNN pre-trained'
             if r['n_visual_unique'] < 10000 else
             'Quy mô lớn, có thể train from scratch hoặc fine-tune thoải mái'))
    if dup_rate >= .10:
        A('- Việc đầu tiên: deduplicate theo dhash rồi mới chia train/val/test')
    if r['leak_groups']:
        A('- Split sẵn có bị leakage → chia lại từ tập đã dedup, stratified theo nhãn')
    if r['imbalance'] >= 1.5:
        A("- Mất cân bằng lớp → class_weight='balanced' hoặc oversampling lớp ít")
    if 'RGBA' in set(ok['mode']):
        A('- Có ảnh RGBA → convert sang RGB trước khi train')
    if ok['mode'].nunique() > 1:
        A(f'- Color mode không thống nhất ({", ".join(map(str, ok["mode"].unique()))}) → ép convert RGB')
    if len(mine) and max(mine.pct_a.max(), mine.pct_b.max()) >= 10:
        A('- Trùng nhiều với dataset khác → nếu gộp nhiều nguồn phải dedup chéo trước khi chia tập')
    A('')

    A('=' * 80)
    A(f'  Thư mục kết quả: {r["out"]}')
    for f in sorted(r['out'].iterdir()):
        A(f'  - {f.name}')
    A('=' * 80)
    return '\n'.join(L)

for r in RES.values():
    txt = build_report(r, CROSS)
    path = r['out'] / f'{r["name"]}_eda_report.txt'
    path.write_text(txt, encoding='utf-8')
    print(f'\n\n{"#"*90}\n# {path}\n{"#"*90}')
    print(txt)

## 9. Bảng so sánh tổng hợp

Gom số liệu quyết định của tất cả dataset vào một file duy nhất ở thư mục gốc: `so_sanh_dataset.txt`. Đây là file đọc để chọn dataset, không phải đọc 3 báo cáo riêng rồi tự so.

Hai cột đáng chú ý nhất:

- **dư thừa %** — ảnh gần trùng nhau trong nội bộ. Cao không hẳn là ảnh rác: nếu dataset chụp cùng một quả ở nhiều góc, hoặc cả bộ chụp cùng một phông nền studio, thì hash thị giác cũng coi là gần trùng. Phải mở `duplicate_samples.png` xem mới kết luận được là nên xóa hay nên gom nhóm khi chia tập.
- **cặp gần trùng khác nhãn %** — trong số các cặp ảnh trông giống nhau, bao nhiêu phần trăm bị gán nhãn khác nhau. Chỉ số này cao nghĩa là nhãn không đi theo vẻ ngoài của ảnh, thường là do dataset định nghĩa độ chín **tương đối theo từng giống chuối**. Gặp trường hợp đó thì đặc trưng màu thủ công sẽ không bám được vào nhãn, và điểm số sẽ thấp dù code không sai.

In [ ]:
L, A = [], None
A = L.append
A('BẢNG SO SÁNH CÁC DATASET – phân loại độ chín của chuối')
A(f'Thời điểm: {datetime.now():%Y-%m-%d %H:%M}   |   MODE = {MODE}')
A('')
A(cmp.to_string(index=False))
A('')

A('CHI TIẾT TỪNG DATASET')
for r in RES.values():
    n = len(r['ok'])
    A(f'\n[{r["name"]}]')
    A(f'  đường dẫn   : {r["root"]}')
    A(f'  lớp         : {", ".join(map(str, r["labels"]))}')
    A(f'  phân bố     : ' + ', '.join(f'{l} {c:,}' for l, c in r['totals'].items()))
    A(f'  dùng được   : {r["n_visual_unique"]:,} ảnh khác nhau '
      f'(~{r["n_visual_unique"]//max(len(r["labels"]),1):,}/lớp)')
    top3 = ', '.join(f'{s.feature} {s.score:.2f}' for _, s in r['sep'].head(3).iterrows())
    A(f'  tách lớp    : {top3}')
    for l in r['labels']:
        g = r['ok'][r['ok'].label == l]
        if len(g):
            A(f'    {l:<14} vàng {g.yellow_frac.mean()*100:>5.1f}%  '
              f'xanh {g.green_frac.mean()*100:>5.1f}%  nâu {g.brown_frac.mean()*100:>5.1f}%')
A('')

A('TRÙNG LẶP CHÉO GIỮA CÁC DATASET')
if len(CROSS):
    for _, c in CROSS.iterrows():
        A(f'- {c.dataset_a} vs {c.dataset_b}: {c.cap_gan_trung:,} cặp gần trùng '
          f'(MD5 {c.trung_md5}, dhash {c.trung_dhash})')
        A(f'    {c.anh_trung_a:,}/{c.n_a:,} ({c.pct_a}%) bên {c.dataset_a}  |  '
          f'{c.anh_trung_b:,}/{c.n_b:,} ({c.pct_b}%) bên {c.dataset_b}')
    mx = max(CROSS.pct_a.max(), CROSS.pct_b.max())
    A(f'- Mức trùng cao nhất giữa hai dataset bất kỳ: {mx}%')
    A('- ' + ('Các dataset độc lập nhau → dùng cái này train, cái kia test '
             'là một phép kiểm chứng sạch' if mx < 1 else
             'Trùng ít, gộp được nhưng phải dedup chéo trước khi chia tập' if mx < 10 else
             'Trùng nhiều, không nên gộp – chọn một cái làm chính'))
else:
    A('- Chỉ có 1 dataset trong lần chạy này')
A('')

A('ĐỌC BẢNG NÀY THẾ NÀO')
A('- Cột "unique sau dedup" mới là lượng dữ liệu thật sự có, không phải cột "tổng ảnh".')
A('- "cặp gần trùng khác nhãn %" cao → nhãn không bám theo màu, đặc trưng thủ công sẽ yếu.')
A('- "có split sẵn = có" thì tái lập được kết quả với tác giả gốc, nhưng vẫn phải xem '
  'cột "leakage split" có bằng 0 không.')
A('- Mất cân bằng từ 1.5x trở lên thì đặt class_weight; từ 3x trở lên nên cân bằng lại dữ liệu.')
A('- Số liệu ở đây chỉ nói về dữ liệu. Trước khi chốt, mở samples_*.png của ứng viên '
  'dẫn đầu để xem nhãn có hợp lý không.')
A('')
A('=' * 80)

txt = '\n'.join(L)
(WORK_DIR / 'so_sanh_dataset.txt').write_text(txt, encoding='utf-8')
print(txt)
print('\nĐã lưu:', WORK_DIR / 'so_sanh_dataset.txt')

## 10. Danh sách file đã sinh ra

Tải các thư mục về từ tab Output của Kaggle.

In [ ]:
for r in RES.values():
    print(f'\n{r["out"]}')
    for f in sorted(r['out'].iterdir()):
        print(f'  {f.stat().st_size/1024:>8.1f} KB  {f.name}')
print(f'\n{FP_DIR}')
for f in sorted(FP_DIR.iterdir()):
    print(f'  {f.stat().st_size/1024:>8.1f} KB  {f.name}')